# Southern Cross Cross-Border Pybind Example

This standalone notebook demonstrates a fictional Southern Cross Banking Group cross-border regulatory slice using only the installed `vannarho-risk-engine` pybind wheel. It does not import helper files or execute source-tree examples.

The notebook has two stages:

1. A simple Australia-scoped balance-sheet and collateral example using AUD pybind primitives.
2. A multi-jurisdiction regulatory summary with synthetic market-risk IMA backtesting and P&L inputs, computed in-notebook because those inputs are not distributed in the wheel-only examples environment.

In [1]:
from pathlib import Path
import importlib.metadata as metadata
import os

_BUILD_ENV_VARS = (
    "VRE_PYBIND_FORCE_BUILD_DIR",
    "VRE_PYBIND_BUILD_DIR",
    "VRE_NOTEBOOK_BUILD_ROOT",
)


def _env_truthy(name: str) -> bool:
    return os.environ.get(name, "").strip() in {"1", "true", "TRUE", "yes", "YES", "on", "ON"}


def import_installed_vre():
    if _env_truthy("VRE_PYBIND_FORCE_BUILD_DIR"):
        raise RuntimeError("This notebook is wheel-only; VRE_PYBIND_FORCE_BUILD_DIR is not supported")
    for name in _BUILD_ENV_VARS[1:]:
        if os.environ.get(name):
            raise RuntimeError(f"This notebook is wheel-only; {name} is not supported")

    try:
        dist = metadata.distribution("vannarho-risk-engine")
    except metadata.PackageNotFoundError as exc:
        raise RuntimeError("Install the vannarho-risk-engine pybind wheel before running this notebook") from exc

    import VRE
    import VREData
    import vre

    dist_root = Path(dist.locate_file("")).resolve()
    for module_name, module in (("VRE", VRE), ("VREData", VREData), ("vre", vre)):
        module_path = Path(getattr(module, "__file__", "")).resolve()
        if any(part == "build" for part in module_path.parts):
            raise RuntimeError(f"{module_name} resolved from a build tree: {module_path}")
        try:
            module_path.relative_to(dist_root)
        except ValueError as exc:
            raise RuntimeError(
                f"{module_name} must resolve from the installed vannarho-risk-engine wheel, not {module_path}"
            ) from exc

    print(f"wheel: {dist.metadata['Name']} {dist.version}")
    print("module: installed VRE pybind wheel")
    return VRE


VRE = import_installed_vre()

wheel: vannarho-risk-engine 0.14.0
module: installed VRE pybind wheel


In [2]:
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format


def _call(obj, method_name, default=None):
    method = getattr(obj, method_name, None)
    if method is None:
        return default
    try:
        return method()
    except Exception:
        return default


def ibor_snapshot(index_name: str) -> dict:
    top_level = VRE.parseIborIndex(index_name)
    object_path = VRE.vred.parsers.objects.parse_ibor_index(index_name)
    top_name = top_level.name()
    object_name = object_path.name()
    if top_name != object_name:
        raise AssertionError(f"Parser paths disagreed for {index_name}: {top_name!r} != {object_name!r}")
    return {
        "input": index_name,
        "name": top_name,
        "tenor": _call(top_level, "tenor", ""),
        "forward_curve_proxy": type(top_level.forwarding_curve()).__name__,
    }


aud_currency = VRE.parseCurrency("AUD")
australia_calendar = VRE.parseCalendar("Australia")
parser_evidence = pd.DataFrame([
    ibor_snapshot("AUD-BBSW-3M"),
    ibor_snapshot("AUD-AONIA"),
])
parser_evidence.insert(0, "currency", str(aud_currency))
parser_evidence["calendar"] = _call(australia_calendar, "name", "Australia")
parser_evidence

,currency,input,name,tenor,forward_curve_proxy,calendar
0,AUD,AUD-BBSW-3M,Bbsw3M Actual/365 (Fixed),3M,YieldTermStructureProxy,Australia settlement
1,AUD,AUD-AONIA,AoniaON Actual/365 (Fixed),1D,YieldTermStructureProxy,Australia settlement


## Stage 1: Australia-Scoped Example

The first stage isolates the Australian parent bank slice. It uses the AUD pybind currency, Australia calendar, and parser bridge evidence for `AUD-BBSW-3M` and `AUD-AONIA`, then computes a compact funding and collateral risk view.

In [3]:
stage1_positions = pd.DataFrame([
    {"book": "AU corporate swaps", "legal_entity": "SCBG_AU_PARENT", "product": "AUD IRS", "notional_aud_m": 520.0, "collateral": "CSA cash", "funding_index": "AUD-AONIA", "spread_bp": 18.0, "risk_weight": 0.20},
    {"book": "AU bank treasury", "legal_entity": "SCBG_AU_PARENT", "product": "Bank bill funding", "notional_aud_m": 310.0, "collateral": "unsecured", "funding_index": "AUD-BBSW-3M", "spread_bp": 42.0, "risk_weight": 0.35},
    {"book": "AU commercial lending hedge", "legal_entity": "SCBG_AU_PARENT", "product": "AUD basis swap", "notional_aud_m": 180.0, "collateral": "CSA cash", "funding_index": "AUD-BBSW-3M", "spread_bp": 29.0, "risk_weight": 0.25},
])

valid_indices = set(parser_evidence["input"])
if not set(stage1_positions["funding_index"]).issubset(valid_indices):
    missing = sorted(set(stage1_positions["funding_index"]) - valid_indices)
    raise AssertionError(f"Funding indices missing parser evidence: {missing}")

stage1_positions["risk_weighted_notional_aud_m"] = (
    stage1_positions["notional_aud_m"] * stage1_positions["risk_weight"]
)
stage1_positions["annual_funding_cost_aud_m"] = (
    stage1_positions["notional_aud_m"] * stage1_positions["spread_bp"] / 10_000.0
)
stage1_positions

,book,legal_entity,product,notional_aud_m,collateral,funding_index,spread_bp,risk_weight,risk_weighted_notional_aud_m,annual_funding_cost_aud_m
0,AU corporate swaps,SCBG_AU_PARENT,AUD IRS,520.00,CSA cash,AUD-AONIA,18.00,0.20,104.00,0.94
1,AU bank treasury,SCBG_AU_PARENT,Bank bill funding,310.00,unsecured,AUD-BBSW-3M,42.00,0.35,108.50,1.30
2,AU commercial lending hedge,SCBG_AU_PARENT,AUD basis swap,180.00,CSA cash,AUD-BBSW-3M,29.00,0.25,45.00,0.52


In [4]:
stage1_summary = (
    stage1_positions
    .groupby("funding_index", as_index=False)
    .agg(
        trades=("product", "count"),
        notional_aud_m=("notional_aud_m", "sum"),
        risk_weighted_notional_aud_m=("risk_weighted_notional_aud_m", "sum"),
        annual_funding_cost_aud_m=("annual_funding_cost_aud_m", "sum"),
    )
    .merge(parser_evidence[["input", "name", "tenor"]], left_on="funding_index", right_on="input", how="left")
    .drop(columns=["input"])
)

assert stage1_summary["notional_aud_m"].sum() == stage1_positions["notional_aud_m"].sum()
stage1_summary

,funding_index,trades,notional_aud_m,risk_weighted_notional_aud_m,annual_funding_cost_aud_m,name,tenor
0,AUD-AONIA,1,520.00,104.00,0.94,AoniaON Actual/365 (Fixed),1D
1,AUD-BBSW-3M,2,490.00,153.50,1.82,Bbsw3M Actual/365 (Fixed),3M


## Stage 2: Multi-Jurisdiction Regulatory Slice

The second stage keeps the same fictional institution and builds a compact cross-border summary across Basel, APRA-style Australian oversight, PRA, EU, US, and MAS views. The market-risk IMA evidence uses synthetic backtesting and P&L vectors produced in this notebook.

In [5]:
jurisdiction_scenarios = pd.DataFrame([
    {"scenario_id": "au_parent_scoped", "regulator": "APRA", "entity_view": "Australia parent bank", "metric": "AUD scoped RWA proxy", "metric_value_aud_m": float(stage1_summary["risk_weighted_notional_aud_m"].sum()), "readiness": "not a filing pack"},
    {"scenario_id": "basel_group_counterparty_cva", "regulator": "BASEL", "entity_view": "Australia parent consolidated group", "metric": "CVA RWA", "metric_value_aud_m": 285.920399, "readiness": "transition reporting"},
    {"scenario_id": "pra_uk_entity_ccr_ccp", "regulator": "PRA", "entity_view": "UK legal entity", "metric": "QCCP trade RWEA", "metric_value_aud_m": 0.0, "readiness": "internal review"},
    {"scenario_id": "eu_entity_imm_flow", "regulator": "EU", "entity_view": "EU legal entity", "metric": "IMM quarter-end RWEA", "metric_value_aud_m": 3.954159, "readiness": "internal review"},
    {"scenario_id": "us_entity_ccr_ccp", "regulator": "US", "entity_view": "US legal entity", "metric": "QCCP trade RWEA", "metric_value_aud_m": 0.0, "readiness": "internal review"},
    {"scenario_id": "mas_entity_imm_flow", "regulator": "MAS", "entity_view": "Singapore legal entity", "metric": "IMM quarter-end RWEA", "metric_value_aud_m": 3.954159, "readiness": "internal review"},
    {"scenario_id": "basel_entity_marketrisk", "regulator": "BASEL", "entity_view": "Basel legal entity trading book", "metric": "SA-MR total RWA", "metric_value_aud_m": 1_891.535185, "readiness": "transition reporting"},
])

jurisdiction_scenarios

,scenario_id,regulator,entity_view,metric,metric_value_aud_m,readiness
0,au_parent_scoped,APRA,Australia parent bank,AUD scoped RWA proxy,257.50,not a filing pack
1,basel_group_counterparty_cva,BASEL,Australia parent consolidated group,CVA RWA,285.92,transition reporting
2,pra_uk_entity_ccr_ccp,PRA,UK legal entity,QCCP trade RWEA,0.00,internal review
3,eu_entity_imm_flow,EU,EU legal entity,IMM quarter-end RWEA,3.95,internal review
4,us_entity_ccr_ccp,US,US legal entity,QCCP trade RWEA,0.00,internal review
5,mas_entity_imm_flow,MAS,Singapore legal entity,IMM quarter-end RWEA,3.95,internal review
6,basel_entity_marketrisk,BASEL,Basel legal entity trading book,SA-MR total RWA,"1,891.54",transition reporting


In [6]:
synthetic_ima_backtesting = pd.DataFrame({
    "day": np.arange(1, 13),
    "hypothetical_pnl_aud_m": [1.2, -0.8, 0.4, -1.7, 0.6, -2.4, 1.1, -0.5, 0.9, -3.1, 0.2, -0.6],
    "actual_pnl_aud_m": [1.1, -0.9, 0.5, -1.9, 0.4, -2.8, 1.0, -0.4, 0.8, -3.6, 0.1, -0.7],
    "var_99_aud_m": [2.0, 2.0, 2.1, 2.0, 2.1, 2.2, 2.1, 2.0, 2.1, 2.3, 2.1, 2.0],
})
synthetic_ima_backtesting["exception"] = synthetic_ima_backtesting["actual_pnl_aud_m"] < -synthetic_ima_backtesting["var_99_aud_m"]
synthetic_ima_backtesting["pnl_basis"] = np.where(
    synthetic_ima_backtesting["exception"],
    "synthetic exception",
    "synthetic observation",
)

ima_exception_count = int(synthetic_ima_backtesting["exception"].sum())
traffic_light = "green" if ima_exception_count <= 4 else "yellow" if ima_exception_count <= 9 else "red"
ima_multiplier = 1.0 if traffic_light == "green" else 1.5 if traffic_light == "yellow" else 2.0

synthetic_ima_backtesting

,day,hypothetical_pnl_aud_m,actual_pnl_aud_m,var_99_aud_m,exception,pnl_basis
0,1,1.20,1.10,2.00,False,synthetic observation
1,2,-0.80,-0.90,2.00,False,synthetic observation
2,3,0.40,0.50,2.10,False,synthetic observation
3,4,-1.70,-1.90,2.00,False,synthetic observation
4,5,0.60,0.40,2.10,False,synthetic observation
5,6,-2.40,-2.80,2.20,True,synthetic exception
6,7,1.10,1.00,2.10,False,synthetic observation
7,8,-0.50,-0.40,2.00,False,synthetic observation
8,9,0.90,0.80,2.10,False,synthetic observation
9,10,-3.10,-3.60,2.30,True,synthetic exception


In [7]:
ima_summary = pd.DataFrame([
    {
        "scenario_id": "basel_entity_marketrisk_synthetic_ima",
        "regulator": "BASEL",
        "entity_view": "Basel legal entity trading book",
        "input_source": "in-notebook synthetic P&L and backtesting",
        "observations": len(synthetic_ima_backtesting),
        "exceptions": ima_exception_count,
        "traffic_light": traffic_light,
        "capital_multiplier": ima_multiplier,
        "stressed_var_proxy_aud_m": round(float(synthetic_ima_backtesting["var_99_aud_m"].mean() * ima_multiplier), 4),
    }
])

assert ima_exception_count == 2
assert traffic_light == "green"
ima_summary

,scenario_id,regulator,entity_view,input_source,observations,exceptions,traffic_light,capital_multiplier,stressed_var_proxy_aud_m
0,basel_entity_marketrisk_synthetic_ima,BASEL,Basel legal entity trading book,in-notebook synthetic P&L and backtesting,12,2,green,1.00,2.08


In [8]:
regulatory_rollup = (
    jurisdiction_scenarios
    .groupby(["regulator", "readiness"], as_index=False)
    .agg(
        scenario_count=("scenario_id", "count"),
        metric_value_aud_m=("metric_value_aud_m", "sum"),
    )
    .sort_values(["regulator", "readiness"])
)

output_bundle = {
    "parser_evidence_rows": len(parser_evidence),
    "stage1_position_rows": len(stage1_positions),
    "jurisdiction_rows": len(jurisdiction_scenarios),
    "ima_exception_count": ima_exception_count,
    "ima_traffic_light": traffic_light,
    "uses_installed_pybind_wheel": True,
}

print(output_bundle)
regulatory_rollup

{'parser_evidence_rows': 2, 'stage1_position_rows': 3, 'jurisdiction_rows': 7, 'ima_exception_count': 2, 'ima_traffic_light': 'green', 'uses_installed_pybind_wheel': True}


,regulator,readiness,scenario_count,metric_value_aud_m
0,APRA,not a filing pack,1,257.50
1,BASEL,transition reporting,2,"2,177.46"
2,EU,internal review,1,3.95
3,MAS,internal review,1,3.95
4,PRA,internal review,1,0.00
5,US,internal review,1,0.00


In [9]:
assert set(parser_evidence["input"]) == {"AUD-BBSW-3M", "AUD-AONIA"}
assert output_bundle["jurisdiction_rows"] >= 6
assert "APRA" in set(jurisdiction_scenarios["regulator"])
assert "BASEL" in set(jurisdiction_scenarios["regulator"])
assert output_bundle["uses_installed_pybind_wheel"] is True
print("Southern Cross cross-border pybind notebook passed")

Southern Cross cross-border pybind notebook passed
